# DHL-MM in 5 Minutes

Sparse Lie algebra multiplication for all five exceptional algebras — up to 993× faster than dense.

[![PyPI](https://img.shields.io/pypi/v/dhl-mm)](https://pypi.org/project/dhl-mm/)

In [ ]:
!pip install -q dhl-mm torch scipy

## All 5 Exceptional Algebras

Precomputed structure constants, loaded in <25ms each.

In [ ]:
import dhl_mm
import numpy as np

for name in ['G2', 'F4', 'E6', 'E7', 'E8']:
    alg = dhl_mm.algebra(name)
    print(f"{name}: dim={alg.dim}, roots={alg.n_roots}, "
          f"sparse={alg.n_structure_constants}, "
          f"compression={alg.dim**3 / alg.n_structure_constants:.1f}x")

## Jacobi Identity Verification

[x,[y,z]] + [y,[z,x]] + [z,[x,y]] = 0 — must hold to machine epsilon.

In [ ]:
import dhl_mm
import numpy as np

e8 = dhl_mm.algebra('E8')
rng = np.random.RandomState(42)
x, y, z = rng.randn(248), rng.randn(248), rng.randn(248)

jac = (e8.bracket(x, e8.bracket(y, z)) +
       e8.bracket(y, e8.bracket(z, x)) +
       e8.bracket(z, e8.bracket(x, y)))

print(f"Jacobi violation: {np.max(np.abs(jac)):.2e}")
print(f"Machine epsilon:  {np.finfo(np.float64).eps:.2e}")
print("Exact!" if np.max(np.abs(jac)) < 1e-12 else "ERROR")

## Benchmark: Sparse vs Dense

In [ ]:
import dhl_mm
import numpy as np
import time

e8 = dhl_mm.algebra('E8')
rng = np.random.RandomState(42)
x, y = rng.randn(248), rng.randn(248)

# Sparse bracket
t0 = time.perf_counter()
for _ in range(10000):
    e8.bracket(x, y)
t_sparse = (time.perf_counter() - t0) / 10000

# Dense
gen = np.array(e8.gen_array)
t0 = time.perf_counter()
for _ in range(1000):
    X = np.einsum('i,iab->ab', x, gen)
    Y = np.einsum('i,iab->ab', y, gen)
    XY = X @ Y - Y @ X
t_dense = (time.perf_counter() - t0) / 1000

print(f"Sparse: {t_sparse*1e6:.0f} us/bracket")
print(f"Dense:  {t_dense*1e6:.0f} us/bracket")
print(f"Speedup: {t_dense/t_sparse:.0f}x")

## Differentiable PyTorch Bracket

In [ ]:
import torch
from equivariant import SparseLieBracket

bracket = SparseLieBracket.from_algebra('E8')
x = torch.randn(248, requires_grad=True)
y = torch.randn(248)

z = bracket(x, y)
z.norm().backward()

print(f"Output: {z.shape}, norm={z.norm().item():.4f}")
print(f"Gradient: {x.grad.shape}, norm={x.grad.norm().item():.4f}")
print("Autograd works!")

## Adjoint Equivariance Verification

For group element g = exp(ad_omega): f(Ad_g x) = Ad_g f(x)

In [ ]:
import dhl_mm
import numpy as np
import torch
from equivariant import EquivariantLieConvLayer
from scipy.linalg import expm

layer = EquivariantLieConvLayer(algebra_name='E8')
layer.eval()

e8 = dhl_mm.algebra('E8')
omega = np.random.randn(248) * 0.01
ad_omega = sum(omega[i] * e8.gen_array[i] for i in range(248))
Ad_g = torch.tensor(expm(ad_omega), dtype=torch.float32)

x = torch.randn(5, 248)
edge_index = torch.tensor([[0,1,2,3,4],[1,2,3,4,0]], dtype=torch.long)

with torch.no_grad():
    out_then_transform = layer(x, edge_index) @ Ad_g.T
    transform_then_out = layer(x @ Ad_g.T, edge_index)

diff = (out_then_transform - transform_then_out).abs().max().item()
print(f"Equivariance error: {diff:.2e}")
print("Equivariant!" if diff < 1e-4 else "NOT equivariant")

## Lattice Gauge Theory

First open-source tooling for E₈ lattice gauge theory.

In [ ]:
from dhl_mm import GaugeLattice

lat = GaugeLattice((6, 6), algebra_name='E8')
lat.hot_start(scale=0.3, seed=42)
result = lat.thermalize(n_sweeps=50, beta=1.0, step_size=0.1, seed=42)

print(f"Sites: {lat.n_sites}, Links: {lat.n_links}, Plaquettes: {lat.n_plaquettes}")
print(f"Initial action: {result['actions'][0]:.2f}")
print(f"Final action:   {result['actions'][-1]:.2f}")
print(f"Acceptance rate: {result['acceptance_rates'][-1]:.1%}")

## RKMK Lie Group Integrators

4th-order Runge-Kutta with Baker-Campbell-Hausdorff bracket corrections.

In [ ]:
import numpy as np
from dhl_mm import RKMKIntegrator

rk = RKMKIntegrator('G2')
rng = np.random.RandomState(99)
H = rng.randn(14) * 0.1
y0 = rng.randn(14) * 0.1

flow = lambda t, y: rk.alg.bracket(H, y)
result = rk.solve(flow, y0, t_span=(0, 1.0), dt=0.01, method='rk4')

k0 = rk.alg.killing_form(y0, y0)
kf = rk.alg.killing_form(result['states'][-1], result['states'][-1])
print(f"Steps: {len(result['times'])}")
print(f"Killing norm: {k0:.6f} -> {kf:.6f}")
print(f"Relative drift: {abs(kf - k0) / abs(k0):.2e}")

---

`pip install dhl-mm` | [GitHub](https://github.com/grapheneaffiliate/DHL-MM) | [PyPI](https://pypi.org/project/dhl-mm/)